In [ ]:
!pip install -q transformers torch

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-2 does not have a pad token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"Loaded {MODEL_NAME} on {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2 on cpu


In [ ]:
def is_python_coding_question(prompt: str) -> bool:
    if not prompt or not prompt.strip():
        return False

    text = prompt.lower().strip()

    # Reject clearly non-programming Python usage
    non_coding_patterns = [
        r"\bis python a snake\b",
        r"\bpython is a snake\b",
        r"\bpython snake\b",
        r"\btype of snake\b",
        r"\breptile\b",
        r"\banimal\b",
        r"\bpet python\b",
        r"\bwhere do pythons live\b",
        r"\bpython habitat\b",
    ]
    if any(re.search(pattern, text) for pattern in non_coding_patterns):
        return False

    # Reject other programming languages explicitly
    non_python_languages = [
        "c++", "java", "javascript", "c#", "ruby", "php", "go", "swift",
        "kotlin", "rust", "typescript", "scala", "perl", "r", "matlab",
        "sql", "html", "css"
    ]
    if any(lang in text for lang in non_python_languages):
        return False

    # Strong coding keywords
    coding_keywords = [
        "python", "code", "coding", "program", "script", "function", "class",
        "method", "debug", "bug", "error", "exception", "traceback", "syntax",
        "loop", "list", "dictionary", "tuple", "set", "decorator", "generator",
        "iterator", "lambda", "recursion", "pandas", "numpy", "matplotlib",
        "flask", "django", "fastapi", "pytest", "unittest", "jupyter",
        "notebook", "import", "def", "return"
    ]

    # Code patterns
    code_patterns = [
        r"\bdef\s+\w+\s*\(",
        r"\bclass\s+\w+\s*[:(]",
        r"\bimport\s+\w+",
        r"\bfrom\s+\w+\s+import\s+",
        r"print\s*\(",
        r"\breturn\b",
        r"\bif\b.+:",
        r"\bfor\b.+\bin\b",
        r"\bwhile\b.+:",
    ]

    if any(re.search(pattern, prompt, flags=re.IGNORECASE | re.DOTALL) for pattern in code_patterns):
        return True

    keyword_hits = sum(1 for kw in coding_keywords if kw in text)

    # Require at least some coding context
    if keyword_hits >= 2:
        return True

    # Allow common question forms when paired with Python
    if "python" in text and any(x in text for x in ["how", "write", "fix", "explain", "implement", "use"]):
        return True

    return False

In [ ]:
def build_python_prompt(user_question: str) -> str:
    return f"""
You are a Python coding assistant.

Answer the question only if it is about Python programming.
Give a short, direct, useful answer.
If code is needed, provide a small Python example.
Do not repeat yourself.
Do not list unrelated attributes or properties.

Question: {user_question}
Answer:
""".strip()

In [ ]:
def answer_python_question(question: str,
                           max_new_tokens: int = 80):
    fallback_message = "I can only answer questions related to Python coding."

    if not is_python_coding_question(question):
        return fallback_message

    prompt = build_python_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                 # deterministic output
            num_beams=3,                     # slightly better than greedy
            no_repeat_ngram_size=3,          # reduce repetition
            repetition_penalty=1.2,          # penalize loops
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            early_stopping=True
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in decoded:
        answer = decoded.split("Answer:", 1)[1].strip()
    else:
        answer = decoded.strip()

    return answer

In [ ]:
question = "How do I reverse a list in Python?"
print(question)

How do I reverse a list in Python?


In [ ]:
print(answer_python_question(question))

You can use the reverse() function to reverse the list. The reverse() method returns a list of all the items in the list, and returns the list as a list. If the list is empty, it will be returned as an empty list. For example:

>>> list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11


In [ ]:
test_questions = [
    "How do I reverse a list in Python?",
    "Write a Python function to check palindrome.",
    "Fix this code: def add(a,b) print(a+b)",
    "Is Python a snake?",
    "What is the capital of France?",
    "Explain pandas read_csv in Python.",
    "Tell me a joke."
]

for q in test_questions:
    print("=" * 80)
    print("Question:", q)
    print("Allowed:", is_python_coding_question(q))
    print("Response:", answer_python_question(q))
    print()

Question: How do I reverse a list in Python?
Allowed: True
Response: You can use the reverse() function to reverse the list. The reverse() method returns a list of all the items in the list, and returns the list as a list. If the list is empty, it will be returned as an empty list. For example:

>>> list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11

Question: Write a Python function to check palindrome.
Allowed: True
Response: You can write a function that checks for palindromes in the same way as a regular function does. For example, you could write:

def checkPalindrome ( self ): return self .palindrome()

You can also write functions that check for Palindrome in other ways, such as by passing a string to the function, or by passing an array to the

Question: Fix this code: def add(a,b) print(a+b)
Allowed: True
Response: If you have a problem with this code, try to fix it by adding the following line to your code:

def add(x,y): return x + y print(x+y)

This will fix the problem. If you don'

In [ ]:
question = "How a python snake can be killed?"
print(answer_python_question(question))

I can only answer questions related to Python coding.


In [ ]:
question = "how does for loop work in c++?"
print(answer_python_question(question))

I can only answer questions related to Python coding.
